# 🏗️ Colab C — PyTorch Classes-Based 3-Layer Neural Network
## Nonlinear Regression using `nn.Module`, `nn.Linear`, `optim`, autograd

### Architecture
```
Input(3) → [Linear(16) + Swish] → [Linear(8) + Swish] → Linear(1)
```
Uses **PyTorch built-in** `nn.Module`, `nn.Linear`, `torch.optim.Adam`


In [ ]:
# ── SECTION 1: Imports ──────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ── SECTION 2: Data ──────────────────────────────────────────────────────────
np.random.seed(42)
N = 1000
X1 = np.random.uniform(-np.pi, np.pi, N)
X2 = np.random.uniform(-np.pi, np.pi, N)
X3 = np.random.uniform(-2, 2, N)
Y  = np.sin(X1) * np.cos(X2) + X3**2 + np.random.randn(N) * 0.1
X  = np.column_stack([X1, X2, X3])

scaler_X = StandardScaler()
scaler_Y = StandardScaler()
X_norm = scaler_X.fit_transform(X)
Y_norm = scaler_Y.fit_transform(Y.reshape(-1,1))

X_t = torch.tensor(X_norm, dtype=torch.float32).to(device)
Y_t = torch.tensor(Y_norm, dtype=torch.float32).to(device)
print(f'Data ready: X={X_t.shape}, Y={Y_t.shape}')

In [ ]:
# ── SECTION 3: Custom Swish Module ──────────────────────────────────────────
class Swish(nn.Module):
    """Custom Swish activation as an nn.Module."""
    def forward(self, x):
        return x * torch.sigmoid(x)

# ── SECTION 4: 3-Layer Neural Network Class ──────────────────────────────────
class ThreeLayerNet(nn.Module):
    """
    3-layer DNN for nonlinear regression.
    Architecture: 3 → 16 → 8 → 1
    Uses nn.Linear + Swish activation + He init.
    """
    def __init__(self, input_dim=3, h1=16, h2=8, output_dim=1):
        super().__init__()
        # Layer definitions
        self.fc1 = nn.Linear(input_dim, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.fc3 = nn.Linear(h2, output_dim)
        self.act  = Swish()

        # He initialization
        for layer in [self.fc1, self.fc2, self.fc3]:
            nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')
            nn.init.zeros_(layer.bias)

    def forward(self, x):
        """Forward pass through all 3 layers."""
        x = self.act(self.fc1(x))  # Layer 1
        x = self.act(self.fc2(x))  # Layer 2
        x = self.fc3(x)            # Output (linear)
        return x

model = ThreeLayerNet().to(device)
print(model)
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── SECTION 5: Loss, Optimizer, Training ─────────────────────────────────────
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)

EPOCHS = 2000
loss_history = []

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()            # clear gradients
    y_pred = model(X_t)              # forward pass
    loss   = criterion(y_pred, Y_t)  # compute loss
    loss.backward()                  # backprop (PyTorch handles chain rule)
    optimizer.step()                 # update weights
    scheduler.step()                 # adjust LR
    loss_history.append(loss.item())

    if epoch % 200 == 0:
        print(f'Epoch {epoch:4d} | Loss: {loss.item():.6f} | LR: {scheduler.get_last_lr()[0]:.5f}')

print(f'\nFinal Loss: {loss_history[-1]:.6f}')

In [ ]:
# ── SECTION 6: Results & Visualization ──────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(loss_history, color='green')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss'); plt.yscale('log')
plt.title('Training Loss — PyTorch nn.Module 3-Layer Net')
plt.grid(True); plt.show()

model.eval()
with torch.no_grad():
    y_pred_np = model(X_t).cpu().numpy()
y_pred_inv = scaler_Y.inverse_transform(y_pred_np)
y_true_inv = scaler_Y.inverse_transform(Y_norm)

plt.figure(figsize=(8, 5))
plt.scatter(y_true_inv[:200], y_pred_inv[:200], alpha=0.5, s=20)
plt.plot([y_true_inv.min(), y_true_inv.max()],
         [y_true_inv.min(), y_true_inv.max()], 'r--', label='Perfect')
plt.xlabel('True y'); plt.ylabel('Predicted y')
plt.title('True vs Predicted — PyTorch Class-Based 3-Layer')
plt.legend(); plt.grid(True); plt.show()
print(f'Final MSE (orig scale): {np.mean((y_true_inv - y_pred_inv)**2):.4f}')